# Data Analyst (итерация 1)

# Data Analyst Report

## Бизнес-задача

Цель данного анализа — исследовать очищенный датасет вакансий для выявления мошеннических объявлений (target: `fraudulent`). Это поможет снизить ручную модерацию и защитить пользователей HR-площадки от скам-постингов.

## Что покажет EDA

- Общий обзор структуры данных
- Анализ распределения целевой переменной
- Корреляции числовых признаков с целевой
- Анализ категориальных признаков и их связь с мошенничеством
- Исследование текстовых колонок по длинам и пропускам

В итоге мы получим глубокое понимание данных, что позволит принимать обоснованные решения при построении модели.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

FIGS = []

DF = pd.read_csv("/Users/nikitayarygin/Documents/projects/ai-agent-gp3/data/processed/cleaned.csv")

print(f"Dataset shape: {DF.shape}")
print("Data types:")
print(DF.dtypes)
print("\nSample data:")
print(DF.head())

Dataset shape: (17880, 36)
Data types:
job_id                                                  float64
title                                                       str
location                                                float64
department                                              float64
company_profile                                             str
description                                                 str
requirements                                                str
benefits                                                    str
telecommuting                                             int64
has_company_logo                                          int64
has_questions                                             int64
industry                                                float64
function                                                    str
fraudulent                                                int64
employment_type_Full-time                                  bool
e

## Обзор датасета

В этой секции мы рассмотрим размер датасета, распределение типов данных, а также выделим числовые, категориальные и текстовые колонки. Также посмотрим базовую статистику по числовым признакам.

In [ ]:
# Определим типы колонок
num_cols = DF.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = DF.select_dtypes(include=["object"]).columns.tolist()

# Предположим, что текстовые колонки - это object с большим количеством уникальных значений и длинными строками
# Для простоты выделим текстовые колонки как те object, у которых средняя длина строки > 50
text_cols = []
for col in cat_cols:
    mean_len = DF[col].dropna().map(len).mean()
    if mean_len > 50:
        text_cols.append(col)

# Категориальные без текстовых
cat_cols = [c for c in cat_cols if c not in text_cols]

print(f"Total columns: {DF.shape[1]}")
print(f"Numerical columns ({len(num_cols)}): {num_cols}")
print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")
print(f"Text columns ({len(text_cols)}): {text_cols}")

# Статистика по числовым колонкам
print("\nNumerical columns summary:")
print(DF[num_cols].describe().T)  # transpose for better view

<string>:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
Total columns: 36
Numerical columns (8): ['job_id', 'location', 'department', 'telecommuting', 'has_company_logo', 'has_questions', 'industry', 'fraudulent']
Categorical columns (2): ['title', 'function']
Text columns (4): ['company_profile', 'description', 'requirements', 'benefits']

Numerical columns summary:
                    count         mean          std         min          25%          50%           75%           max
job_id            17880.0  8940.500000  5158.582478  179.790000  4470.750000  8940.500000  

## Распределение целевой переменной `fraudulent`

In [ ]:
target_counts = DF["fraudulent"].value_counts()
print("Target value counts:")
print(target_counts)

imbalance_ratio = target_counts.min() / target_counts.max()
print(f"Imbalance ratio (minority/majority): {imbalance_ratio:.4f}")

# График 1: bar chart распределения target
fig1 = px.bar(x=target_counts.index.astype(str), y=target_counts.values,
              labels={"x": "fraudulent", "y": "Count"},
              title="Распределение целевой переменной fraudulent (count)")
FIGS.append(fig1)

# График 2: pie chart долей
fig2 = px.pie(values=target_counts.values, names=target_counts.index.astype(str),
              title="Доли классов fraudulent")
FIGS.append(fig2)

Target value counts:
fraudulent
0    17014
1      866
Name: count, dtype: int64
Imbalance ratio (minority/majority): 0.0509


## Корреляции числовых признаков с целевой переменной

In [ ]:
# Корреляция числовых признаков с target
corrs = DF[num_cols].corrwith(DF["fraudulent"]).sort_values(key=lambda x: x.abs(), ascending=False)
print("Корреляция числовых признаков с fraudulent (по абсолютному значению):")
print(corrs)

# График 3: heatmap корреляций числовых признаков
corr_matrix = DF[num_cols].corr()
fig3 = px.imshow(corr_matrix, text_auto=True, aspect="auto",
                 title="Корреляционная матрица числовых признаков")
FIGS.append(fig3)

# График 4: bar chart абсолютных корреляций с target
fig4 = px.bar(x=corrs.index, y=corrs.abs(),
              labels={"x": "Feature", "y": "|Correlation with fraudulent|"},
              title="Абсолютные корреляции числовых признаков с fraudulent")
FIGS.append(fig4)

# График 5: распределение топ-2 числовых признаков по классам target
top2_num = corrs.index[:2].tolist()
for col in top2_num:
    fig = go.Figure()
    for cls in sorted(DF["fraudulent"].unique()):
        fig.add_trace(go.Histogram(x=DF.loc[DF["fraudulent"]==cls, col],
                                   name=f"fraudulent={cls}", opacity=0.6))
    fig.update_layout(barmode='overlay', title=f"Распределение {col} по классам fraudulent",
                      xaxis_title=col, yaxis_title="Count")
    FIGS.append(fig)
    print(f"Распределение по классам для {col} построено.")

Корреляция числовых признаков с fraudulent (по абсолютному значению):
fraudulent          1.000000
has_company_logo   -0.261971
industry           -0.111830
has_questions      -0.091627
job_id              0.079491
department         -0.049204
location           -0.047613
telecommuting       0.034523
dtype: float64
Распределение по классам для fraudulent построено.
Распределение по классам для has_company_logo построено.


## Анализ категориальных признаков

In [ ]:
# Выберем топ-3 категориальных по количеству уникальных значений
cat_cardinality = {col: DF[col].nunique() for col in cat_cols}
cat_top3 = sorted(cat_cardinality, key=cat_cardinality.get, reverse=True)[:3]
print(f"Топ-3 категориальных признаков по уникальным значениям: {cat_top3}")

for col in cat_top3:
    mean_target_by_cat = DF.groupby(col)["fraudulent"].mean().sort_values(ascending=False).head(10)
    print(f"\nСредний таргет fraudulent по топ-10 категориям в {col}:")
    print(mean_target_by_cat)

    # Графики 6-8: bar chart mean target rate по топ-10 категориям
    fig = px.bar(x=mean_target_by_cat.index.astype(str), y=mean_target_by_cat.values,
                 labels={"x": col, "y": "Mean fraudulent rate"},
                 title=f"Средний уровень мошенничества по топ-10 категориям {col}")
    FIGS.append(fig)

Топ-3 категориальных признаков по уникальным значениям: ['title', 'function']

Средний таргет fraudulent по топ-10 категориям в title:
title
Offshore Construction Superintendent    1.0
KMC                                     1.0
Receptionist and Office Assistant       1.0
Typist / Data Processing Clerical       1.0
Accounting Clerk/ $23                   1.0
Full/PartTime Data Entry Work           1.0
Accounting Clerk($20/hr)                1.0
Accounting Clerk - $25                  1.0
Structural Designer                     1.0
Data Processing Agent                   1.0
Name: fraudulent, dtype: float64

Средний таргет fraudulent по топ-10 категориям в function:
function
Administrative          0.188889
Financial Analyst       0.151515
Accounting/Auditing     0.136792
Distribution            0.125000
Other                   0.098462
Finance                 0.087209
Engineering             0.083828
Business Development    0.057018
Advertising             0.055556
Project Management  

## Анализ текстовых колонок

In [ ]:
# Рассчитаем длину текста (word count) для каждой текстовой колонки
for col in text_cols:
    DF[f"{col}_word_count"] = DF[col].fillna("").map(lambda x: len(str(x).split()))

for col in text_cols:
    wc_col = f"{col}_word_count"
    mean_wc_by_target = DF.groupby("fraudulent")[wc_col].mean()
    nan_rate = DF[col].isna().mean()
    print(f"\nТекстовая колонка: {col}")
    print(f"Среднее количество слов по классам fraudulent:")
    print(mean_wc_by_target)
    print(f"Доля пропусков: {nan_rate:.4f}")

    # График 9: распределение длины текста по target
    fig9 = go.Figure()
    for cls in sorted(DF["fraudulent"].unique()):
        fig9.add_trace(go.Histogram(x=DF.loc[DF["fraudulent"]==cls, wc_col],
                                    name=f"fraudulent={cls}", opacity=0.6))
    fig9.update_layout(barmode='overlay', title=f"Распределение длины текста ({col}) по классам fraudulent",
                       xaxis_title="Word count", yaxis_title="Count")
    FIGS.append(fig9)

    # График 10: bar chart доли пропусков по target
    nan_rate_by_target = DF.groupby("fraudulent")[col].apply(lambda x: x.isna().mean())
    fig10 = px.bar(x=nan_rate_by_target.index.astype(str), y=nan_rate_by_target.values,
                   labels={"x": "fraudulent", "y": "NaN rate"},
                   title=f"Доля пропусков в {col} по классам fraudulent")
    FIGS.append(fig10)


Текстовая колонка: company_profile
Среднее количество слов по классам fraudulent:
fraudulent
0    95.650523
1    31.709007
Name: company_profile_word_count, dtype: float64
Доля пропусков: 0.1850

Текстовая колонка: description
Среднее количество слов по классам fraudulent:
fraudulent
0    171.041378
1    158.748268
Name: description_word_count, dtype: float64
Доля пропусков: 0.0001

Текстовая колонка: requirements
Среднее количество слов по классам fraudulent:
fraudulent
0    79.031856
1    58.407621
Name: requirements_word_count, dtype: float64
Доля пропусков: 0.1508

Текстовая колонка: benefits
Среднее количество слов по классам fraudulent:
fraudulent
0    30.018338
1    29.451501
Name: benefits_word_count, dtype: float64
Доля пропусков: 0.4034


## Сводка

В данном EDA мы подробно рассмотрели структуру очищенного датасета, распределение целевой переменной, выявили числовые признаки с наибольшей корреляцией к мошенничеству, проанализировали категориальные признаки и их связь с таргетом, а также исследовали текстовые колонки по длинам и пропускам.

Дальнейшие шаги — использовать эти инсайты для построения и настройки моделей классификации с приоритетом на F1 и recall класса 1 при контроле precision.